In [ ]:
%pip install -r annotation_requirements.txt 

In [1]:
import webdataset as wds
import webdataset as wds
import loguru
import os
logger = loguru.logger

from ipywidgets import (Button, Dropdown, HTML, HBox, VBox, FileUpload, Output, Layout, RadioButtons, Text, Checkbox, ToggleButtons)
from IPython.display import Audio, display, clear_output
from ipywidgets.widgets.widget_media import Audio as AudioWidget

Please execute the cell below to set your first name.

In [4]:
# add a textbox widget with a submit button to get the user's name
first_name_text_box = Text(description='First Name:')
# add submit button
FIRST_NAME = ""
def set_first_name(b):
    global FIRST_NAME
    FIRST_NAME = first_name_text_box.value.lower()
    logger.info(f"Set first name to {FIRST_NAME}")

submit_button_name = Button(description='Submit')
submit_button_name.on_click(set_first_name)
display(first_name_text_box)
display(submit_button_name)


Text(value='', description='First Name:')

Button(description='Submit', style=ButtonStyle())

2024-09-23 10:20:31.724 | INFO     | __main__:set_first_name:8 - Set first name to nobody


# Annotation Instructions

First, we thank you for agreeing to helping us with providing your preference annotations! Your expertise in phonetic transcription is valuable and appreciated. Please read the following instructions carefully.

**Options**. The annotation task will comprise 50 datapoints of the form: (audio, transcript1, transcript2). You'll listen to the audio and then pick one of four options:

1. TranscriptA > TranscriptB 
2. TranscriptA < TranscriptB 
3. TranscriptA = TranscriptB (both are equally good)
4. TranscriptA = TranscriptB (both are equally poor)

You may replay the audio as many times as you like. You can also change the playback speed of the audio using the "kebab menu" (three dots).

**Justification**. When you select (1), you will then select which word(s) in the transcript were better represented in TranscriptA compared to TranscriptB. (Analogous for Option 2). No need to be exhaustive here, just select some of the word(s) that seemed most well represented to you (relative to the other transcript).

**Avoid abstaining (options 3 and 4) if possible**. Try to select option (1) or (2) when possible, only resorting to (3) or (4) when you find it impossible to pick between the two. (Ideally, no more than 10 samples should have the (3) or (4) option). When both TranscriptA and TranscriptB have problems, try to select the transcript that has fewer problems.

**Transcript spacing**. When comparing the two transcripts, don't use the whitespace segmentation of the transcripts in informing your decision. The spaces are automatically inserted to improve readability of the transcripts, and there may occasionally be some spacing errors in TranscriptA relative to TranscriptB (or vice-versa). For example, one of the transcripts may segment the phrase "the car" into "ðək ɑr" instead of "ðə kɑr". Please try to ignore these spacing discrepancies in making your determination, instead focusing on whether the phonetic segments accurately represent the speech audio. For example, if TranscriptA = "θi kɛr" while TranscriptB = "ðək ɑr", you should prefer TranscriptB since it has more accurate phonetic segments (assuming a Standard American English pronunciation), and ignore the fact that the space is inserted after the "k" in "kɑr" rather than after the vowel in "ðə". Note that the spacing can also result in affricates ("d͡ʒ" or "t͡s") being broken up ("d ʒ" or "t s"), so if you clearly hear an affricate but don't see a tie bar in the transcript, the affricate may well be represented but (inadvertently) broken up between two words.

**Going back to previous annotations**. The interface contains back and forward buttons. When you hit the back button to navigate to the previous sample, you'll see that your prior annotation is stored. If you want to select a different option (1-4), you can do so and hit submit. If you want to leave it as is, you can just hit the "Forward" button again. When you use "Back", The "Submit" button will be disabled unless you change your selection. 

**Display**. On rare occassions, some of the transcripts can be a bit long and rendered incorrectly. Ensuring that the window is full-screen and using a larger external monitor (if possible) should mitigate this.

When you are ready, execute the next cell to begin. At the bottom of the cell, the interface for performing the annotations will appear. Your progress will be saved as you work through the annotations, so feel free to take breaks. 

In [7]:
import random
import numpy as np
from typing import Tuple
import json
import ipywidgets as widgets
import random

from itertools import islice

ANNOTATION_DIR = "annotation_wds"
FILENAME = "ar_eg-da_dk-ga_ie-ml_in-sd_in"
annotation_fname = f"{ANNOTATION_DIR}/{FILENAME}.tar"

# open the json file, if it exists. 
    # we can skip the samples that have already been annotated
# if os.path.exists(f"{ANNOTATION_TAR_DIR}/{NEXT_LANGUAGE}_{FIRST_NAME}.json"):

def load_json_lines(fname):
    data = []
    with open(fname, "r") as f:
        for line in f:
            data.append(json.loads(line))
    return data
if os.path.exists(f"{ANNOTATION_DIR}/{FILENAME}_{FIRST_NAME}.json"):
    annotated_samples = load_json_lines(f"{ANNOTATION_DIR}/{FILENAME}_{FIRST_NAME}.json")
    annotations = annotated_samples
    annotated_keys = set([x["key"] for x in annotated_samples])
    dataset = list(wds.WebDataset(annotation_fname).decode().to_tuple("npy", "txt", '__key__'))
    num_total = len(dataset)
    dataset_keys = set([x[2] for x in dataset])
    num_annotated = len(annotated_keys.intersection(dataset_keys))
    # get the samples that have not been annotated -- dataset_keys - annotated_keys
    # dataset = [x for x in dataset if x[2] not in annotated_keys]
else:
    logger.info(f"No annotated samples found.")
    dataset = list(wds.WebDataset(annotation_fname).decode().to_tuple("npy", "txt", '__key__'))
    num_total = len(dataset)
    num_annotated = 0
    annotations = [] # store annotations for each sample

current_i = num_annotated
# TODO: need to update annotations list with the annotations that have already been done
layout = widgets.Layout(width='auto', justify_content='center')
progress = widgets.IntProgress(
    value=num_annotated,
    min=0,
    max=num_total,
    step=1,
    description='Completed:',
    bar_style='', # 'success', 'info', 'warning', 'danger' or ''
    orientation='horizontal'
)
progress_box = widgets.Box(children=[progress], layout=layout)
json_entry = {}

def save_annotations_to_json(annotations, fname):
    with open(fname, "w") as f:
        for annotation in annotations:
            f.write(json.dumps(annotation) + "\n")

def load_gt_transcript(sample: Tuple[np.array, str, str]):
    audio, transcript, _ = sample
    transcript = transcript.split('\n')[0]
    return transcript

def load_predicted_transcript(sample: Tuple[np.array, str, str]):
    audio, transcripts, _ = sample
    predicted_transcript = ' '.join(eval(transcripts.split("\n")[1]))
    return predicted_transcript

def create_radio_btn(curr_sample, radio_state = None):
    gt_transcript = load_gt_transcript(curr_sample).strip()
    predicted_transcript = load_predicted_transcript(curr_sample).strip()
    global json_entry
    json_entry = {
        "gt_transcript": gt_transcript,
        "predicted_transcript": predicted_transcript,
        "key": curr_sample[2]
    }
    # flip a coin to decide whether to show the gt or predicted transcript first
    if random.choice([True, False]):
        entry_one, entry_two = predicted_transcript, gt_transcript
    else:
        entry_one, entry_two = gt_transcript, predicted_transcript
    # TODO: add some idk options
    options = [entry_one, entry_two, "Unselected", "Unsure (both transcripts are equally poor)", "Unsure (both transcripts are equally good)"]
    radio_button = RadioButtons(
        options=options,
        description=f'Which transcript do you prefer:',
        value = (radio_state if radio_state in options else "Unselected"),
        disabled=False,
        layout=widgets.Layout(width='auto', white_space='normal')
    )
    radio_button.style = {'description_width': 'initial'}
    radio_button.observe(radio_btn_onclick, names='value')
    return radio_button

def load_sample(index):
    global sample_box, submit_button
    sample = dataset[index]
    if index < len(annotations):
        annotation_state = annotations[index]
        next_sample_box = load_sample_box(sample, annotation_state)
    else:
        next_sample_box = load_sample_box(sample)
    sample_box.children = next_sample_box.children
    submit_button.disabled = True

def submit_sample_action(sender):
    global progress
    global sample_box
    global submit_box
    global submit_button
    global json_entry
    global current_i
    for child in sample_box.children:
        child.close()
    progress.value += 1

    # TODO: need to log the results of the checkboxes and the radio button
    radio_button = sample_box.children[1]
    radio_selected = radio_button.value

    selected_words = []
    if radio_selected not in ["Unselected", "Unsure (both transcripts are equally poor)", "Unsure (both transcripts are equally good)"]:
        checkboxes = sample_box.children[2]
        if checkboxes:
            for checkbox in checkboxes.children:
                if checkbox.value:
                    selected_words.append(checkbox.description)
    json_entry["selected_words"] = selected_words
    json_entry["radio_selected"] = radio_selected
    # with open(f"{ANNOTATION_DIR}/{FILENAME}_{FIRST_NAME}.json", "a") as f:
    #     f.write(json.dumps(json_entry) + "\n")
    # append to annotations list, if current_i == len(annotations), then we can just append
    # otherwise, we need to update the entry
    if current_i == len(annotations):
        annotations.append(json_entry)
    else:
        annotations[current_i] = json_entry
    save_annotations_to_json(annotations, f"{ANNOTATION_DIR}/{FILENAME}_{FIRST_NAME}.json")
    
    if current_i < num_total - 1:
        current_i += 1
        load_sample(current_i)
        update_navigation_buttons(current_i)
    else:
        logger.info("No more samples to annotate.")
        # clear the display
        progress_box.close()
        sample_box.close()
        submit_box.close()
        submit_button.close()

def navigate_back(sender):
    global current_i
    if current_i > 0:
        current_i -= 1
        load_sample(current_i)
        update_navigation_buttons(current_i)

def navigate_forward(sender):
    global current_i
    # don't allow going beyond the last sample
    if current_i < num_total - 1 and current_i < progress.value:
        current_i += 1
        load_sample(current_i)
        update_navigation_buttons(current_i)

center_layout = widgets.Layout(display='flex', 
                               justify_content='center', 
                               align_items='center')

def create_navigation_buttons(current_i):
    back_button = Button(description='Back', disabled=True if current_i == 0 else False)
    back_button.on_click(navigate_back)
    forward_button = Button(description='Forward', disabled=True if (current_i == num_total - 1 or current_i == progress.value) else False)
    forward_button.on_click(navigate_forward)
    return back_button, forward_button

def create_navigation_box(current_i):
    back_button, forward_button = create_navigation_buttons(current_i)
    return widgets.HBox([back_button, forward_button], layout=center_layout)

def update_navigation_buttons(current_i):
    global navigation_box
    buttons = create_navigation_buttons(current_i)
    navigation_box.children =[buttons[0], buttons[1]]

def load_sample_box(sample, annotation_state = None):
    audio_display = Audio(data=sample[0], autoplay=False, rate=16000)
    audio_widget = widgets.HTML(audio_display._repr_html_())
    audio_widget.layout = widgets.Layout(padding='150px 0 0 0')
    if annotation_state:
        radio_btn = create_radio_btn(sample, annotation_state['radio_selected'])
    else:
        radio_btn = create_radio_btn(sample)
    sample_box = widgets.VBox(children=[audio_widget, radio_btn], layout=center_layout)
    return sample_box
pref_words_box = None

def create_submit_button():
    button = Button(
        description='Submit',
        justify_content='center',
        disabled=True
    )
    button.on_click(submit_sample_action)
    return button


def radio_btn_onclick(sender):
    # TODO: add checkboxes for which words are preferred in the preferred transcript
    global sample_box
    global pref_words_box
    global submit_box
    global submit_button
    if pref_words_box:
        pref_words_box.close()
    # if submit_box:
    #     submit_box.close()

    # if the radio button value is not unselected, then we can enable the submit button 

    
    radio_btn = sample_box.children[1]
    radio_selected = radio_btn.value
    submit_button.disabled = (radio_selected == "Unselected")
    words = radio_selected.strip().split(" ")
    checkbox_children = []
    if radio_selected not in ["Unselected", "Unsure (both transcripts are equally poor)", "Unsure (both transcripts are equally good)"]:
        for word in words:
            if word.strip() != "":
                checkbox = widgets.Checkbox(
                    value=False,
                    description=word,
                    disabled=False
                )
                checkbox_children.append(checkbox)
        pref_words_box = widgets.Box(children=checkbox_children)
        pref_words_box.layout.display = 'flex'
        pref_words_box.layout.flex_flow = 'column'
        pref_words_box.layout.align_items = 'stretch'
        sample_box.children = list(sample_box.children[0:2]) +  [pref_words_box]

    # submit_btn = create_submit_button()
    # submit_box = widgets.Box(children=[submit_btn], layout=center_layout)

# if there are no more samples to annotate, then we don't have to display anything
if progress.value == num_total:
    logger.info("No more samples to annotate.")
    # clear the display
else:
    sample = dataset[current_i]
    submit_button = create_submit_button()
    submit_box = widgets.Box(children=[submit_button], layout=center_layout)
    sample_box = load_sample_box(sample)
    navigation_box = create_navigation_box(current_i)
    display(sample_box)
    display(submit_box)
    display(progress_box)
    display(navigation_box)

2024-09-25 23:29:55.948 | INFO     | __main__:<module>:35 - No annotated samples found.


Box(children=(Button(description='Submit', disabled=True, style=ButtonStyle()),), layout=Layout(align_items='c…

Box(children=(IntProgress(value=0, description='Completed:', max=50),), layout=Layout(justify_content='center'…